In [5]:
import os 
import pandas as pd
import numpy as np
import nltk

Dir_project = '/work/desai-lab/ABC_dataset'
Dir_derivatives = os.path.join(Dir_project,'derivatives')
Dir_smooth = os.path.join(Dir_derivatives,'afni-smooth')
Dir_fmriprep = os.path.join(Dir_derivatives,'fmriprep')
Dir_raw = '/work/desai-lab/ABCAI/raw/'

df = pd.read_csv('/work/desai-lab/xuanyang/Project/ABC/fMRI/fmri_pass/master_subjects_fmri_passage_complete.csv')
df = df[['subID','sesID','story_english','Hearing_max','ACC','age_dayofday1testing']]
df

,subID,sesID,story_english,Hearing_max,ACC,age_dayofday1testing
0,sub-1006,visit01,Brandi_TaleofTwoCountries.wav,0.0,1.00,61.0
1,sub-1007,visit01,Jack_ProfessionalDecline.wav,1.0,0.50,69.0
2,sub-1008,visit01,Jacob_FairyTree.wav,1.0,0.75,77.0
3,sub-1009,visit01,Jacob_HowCouldYou.wav,4.0,1.00,75.0
4,sub-1011,visit01,Jacob_TigerandLion.wav,1.0,0.75,66.0
...,...,...,...,...,...,...
109,sub-1183,visit01,Brandi_Roswell.wav,0.0,0.75,21.0
110,sub-1185,visit01,Allison_MetaphorsPronouns.wav,1.0,1.00,20.0
111,sub-1187,visit01,Allison_BulliesPronouns.wav,0.0,1.00,22.0
112,sub-1194,visit01,Jack_ProfessionalDecline.wav,1.0,0.25,46.0


In [10]:
dfs_timing= pd.read_csv('/work/desai-lab/xuanyang/Project/ABC/fMRI/fmri_pass/ABC_passage_timings.csv')
dfs_timing

,raw,word,onset,offset,audio,PoS_tag_full,AoA_LWV,OLD20,PLD20,Nsenses_WordNet,...,Sem_N,Sem_Diversity,Consistency_Token_FF,Consistency_Type_FF,Consistency_Token_FB,Consistency_Type_FB,ROOT1_FamSize,Valence_NRC,Arousal_NRC,Dominance_NRC
0,Koko 1.700 2.300,Koko,1.7,2.3,Jacob_Koko,NNP,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,did 4.500 4.900,did,4.5,4.9,Jacob_Koko,VBD,NaN,1.25,1.00,13.0,...,9250.0,2.126914,0.832616,0.841040,0.866084,0.867352,6.0,NaN,NaN,NaN
2,you 4.900 5,you,4.9,5.0,Jacob_Koko,PRP,NaN,1.85,1.00,0.0,...,8468.0,2.030400,0.609223,0.606674,0.433655,0.391659,1.0,NaN,NaN,NaN
3,know 5 5.200,know,5.0,5.2,Jacob_Koko,VBP,4.685078,1.65,1.00,12.0,...,8162.0,1.980890,0.606487,0.588867,0.238291,0.228893,20.0,0.765,0.361,0.704
4,that 5.200 5.500,that,5.2,5.5,Jacob_Koko,IN,NaN,1.65,1.25,0.0,...,9891.0,2.350962,0.339747,0.301681,0.492444,0.468225,1.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43398,to 355 355.400,to,355.0,355.4,Allison_MetaphorsRepeatedNames,TO,NaN,1.15,1.00,0.0,...,9796.0,2.372819,0.405733,0.403407,0.451664,0.439392,10.0,NaN,NaN,NaN
43399,look 355.400 355.600,look,355.4,355.6,Allison_MetaphorsRepeatedNames,VB,4.208655,1.40,1.10,14.0,...,8543.0,2.017486,0.717252,0.700258,0.647490,0.632982,7.0,0.719,0.429,0.615
43400,for 355.600 355.800,for,355.6,355.8,Allison_MetaphorsRepeatedNames,IN,NaN,1.30,1.00,0.0,...,9733.0,2.380212,0.491213,0.484976,0.435592,0.426956,3.0,NaN,NaN,NaN
43401,another 355.800 355.900,another,355.8,355.9,Allison_MetaphorsRepeatedNames,DT,4.648120,2.55,2.70,1.0,...,9852.0,2.291655,0.646291,0.640534,0.596845,0.596409,1.0,NaN,NaN,NaN


In [7]:
Dir_working = '/work/desai-lab/xuanyang/Project/ABC/fMRI/fmri_pass/'
FWHM = 6
model1 = 'Valence_NRC'
model2 = '6mot4polortsm6pca5WM5CSFmotCensor'

resultDir = os.path.join(Dir_working,model1,'results',model2)

Dir_firstlevel = os.path.join(resultDir,'firstlevel')
Dir_secondlevel = os.path.join(resultDir,'secondlevel')
if not os.path.exists(resultDir):
    os.makedirs(resultDir,exist_ok=True)
if not os.path.exists(Dir_firstlevel):
    os.mkdir(Dir_firstlevel)
if not os.path.exists(Dir_secondlevel):
    os.mkdir(Dir_secondlevel)


In [12]:

import subprocess


for i,subID,sesID in zip(df.index,df.subID,df.sesID):

    df.loc[i,'statImg'] = os.path.join(Dir_smooth,subID,'ses-'+sesID,'func','{}_ses-{}_task-Passage_space-MNI152NLin2009cAsym_res-2_desc-sm{}_bold.nii.gz'.format(subID,sesID,FWHM))
    
    
    df.loc[i,'mask_T1'] = os.path.join(Dir_fmriprep,subID,'ses-'+sesID,'anat','{}_ses-{}_space-MNI152NLin2009cAsym_res-2_desc-brain_mask.nii.gz'.format(subID,sesID))
    df.loc[i,'mask_CSF'] = os.path.join(Dir_fmriprep,subID,'ses-'+sesID,'anat','{}_ses-{}_space-MNI152NLin2009cAsym_res-2_label-CSF_probseg.nii.gz'.format(subID,sesID))
    df.loc[i,'mask_WM'] = os.path.join(Dir_fmriprep,subID,'ses-'+sesID,'anat','{}_ses-{}_space-MNI152NLin2009cAsym_res-2_label-WM_probseg.nii.gz'.format(subID,sesID))
    
    if not os.path.exists(df.loc[i,'mask_T1']):
        df.loc[i,'mask_T1'] = os.path.join(Dir_fmriprep,subID,'anat','{}_space-MNI152NLin2009cAsym_res-2_desc-brain_mask.nii.gz'.format(subID,sesID))
        if not os.path.exists(df.loc[i,'mask_T1']):
            print(subID,sesID,'mask_T1')   
    if not os.path.exists(df.loc[i,'mask_CSF']):
        df.loc[i,'mask_CSF'] = os.path.join(Dir_fmriprep,subID,'anat','{}_space-MNI152NLin2009cAsym_res-2_label-CSF_probseg.nii.gz'.format(subID,sesID))
        if not os.path.exists(df.loc[i,'mask_CSF']):
            print(subID,sesID,'mask_CSF')   
   
    if not os.path.exists(df.loc[i,'mask_WM']):
        df.loc[i,'mask_WM'] = os.path.join(Dir_fmriprep,subID,'anat','{}_space-MNI152NLin2009cAsym_res-2_label-WM_probseg.nii.gz'.format(subID,sesID))
        if not os.path.exists(df.loc[i,'mask_WM']):
            print(subID,sesID,'mask_WM')  
    
    if os.path.exists(df.loc[i,'mask_CSF']):
        
        input_dataset =  df.loc[i,'mask_CSF']
        output_dataset = df.loc[i,'mask_CSF'].replace('.nii.gz','_mask')
        
        df.loc[i,'mask_CSF'] = output_dataset
    
    if os.path.exists(df.loc[i,'mask_WM']):
        
        input_dataset =  df.loc[i,'mask_WM']
        output_dataset = df.loc[i,'mask_WM'].replace('.nii.gz','_mask')
        
        df.loc[i,'mask_WM'] = output_dataset
    


sub-1011 visit01 mask_T1
sub-1011 visit01 mask_CSF
sub-1011 visit01 mask_WM
sub-1063 visit01 mask_T1
sub-1063 visit01 mask_CSF
sub-1063 visit01 mask_WM
sub-1110 visit01 mask_T1
sub-1110 visit01 mask_CSF
sub-1110 visit01 mask_WM


In [36]:
mask_Ventricles = '/work/desai-lab/xuanyang/Project/ABC/fMRI/fmri_pass/English_Foreign_fmriprep/codes/Mask_CSF_ventricles2_rs.nii'
MNItmp = '/work/apps/AFNI/linux_openmp_64/MNI152_2009_template_SSW.nii.gz'
for subID, story in zip(df.subID,df.story_english):
    df_subj = df.loc[(df.subID==subID)&(df.sesID=='visit01')]
    
    for sesID, imgFile,mask_T1,mask_CSF,mask_WM in zip(df_subj.sesID,df_subj.statImg,df_subj.mask_T1,df_subj.mask_CSF,df_subj.mask_WM):

        if not os.path.exists(os.path.join(Dir_firstlevel,subID)):
            os.makedirs(os.path.join(Dir_firstlevel,subID))

        output_1st_subj = os.path.join(Dir_firstlevel,subID,sesID)

        if not os.path.exists(output_1st_subj):
            os.makedirs(output_1st_subj)

        scriptFile = os.path.join(output_1st_subj,'Batch_firstlevel_{}_{}.sh'.format(subID,sesID))

        path_confounds = os.path.join(Dir_fmriprep,subID,'ses-'+sesID,'func','{}_ses-{}_task-Passage_desc-confounds_timeseries.tsv'.format(subID,sesID))
        if not os.path.exists(path_confounds):
            print('check:{}-{}.'.format(subID,sesID))
            continue

        df_confounds = pd.read_csv(path_confounds,sep='\t')
            
        
        df_motions = df_confounds[['trans_x','trans_y','trans_z','rot_x','rot_y','rot_z']].copy()
        path_motion = os.path.join(output_1st_subj,'motions_6.1D')
        #if not os.path.exists(path_motion):
        df_motions.to_csv(path_motion,index=False,header=False,sep=' ')

        list_outlier = [outlier for outlier in df_confounds.columns if 'motion_outlier' in outlier]
        df_confounds['motion_outlier'] = 1-df_confounds[list_outlier].sum(axis=1)
        
        path_motion_outlier = os.path.join(output_1st_subj,'motions_outlier.1D')
        df_confounds['motion_outlier'].to_csv(path_motion_outlier,index=False,header=False,sep=' ')
        Dir_timing = os.path.join(Dir_raw,subID,'func')

        
        
        audio = story.replace('.wav','')
        if audio == 'Jacob_Bradford':
            audio = 'Jacob_BradfordBull'
        df_tmp = dfs_timing.loc[dfs_timing.audio==audio]

        lines_regressors = []
        df_tmp_regressor = df_tmp.loc[~df_tmp.Valence_NRC.isna(),['onset','Valence_NRC']]
        df_tmp_missing = df_tmp.loc[df_tmp.Valence_NRC.isna(),'onset'].astype(str).values
        timingFile_regressor = os.path.join(output_1st_subj,'{}_ses-{}_passage-{}_regressors.1D'.format(subID,sesID,audio))
        timingFile_missing = os.path.join(output_1st_subj,'{}_ses-{}_passage-{}_missing.1D'.format(subID,sesID,audio))

        for i,onset,score in zip(df_tmp_regressor.index,df_tmp_regressor.onset,df_tmp_regressor.Valence_NRC):
            values = [str(score)]
            line = str(onset) + '*' + ','.join(values)
            lines_regressors.append(line)
            with open(timingFile_regressor,'w') as f:
                f.writelines('\n'.join(lines_regressors))
            with open(timingFile_missing,'w') as f:
                f.writelines('\n'.join(df_tmp_missing))
        
        
        with open(scriptFile, "w") as f:
            f.write("#!/bin/tcsh\n")
            f.write("/work/apps/AFNI/linux_openmp_64/afni_proc.py -subj_id {} \\\n".format(subID))
            f.write("-scr_overwrite -blocks mask regress \\\n")
            f.write("-mask_import CSF {} \\\n".format(mask_CSF))
            f.write("-mask_import WM {} \\\n".format(mask_WM))
            #f.write("-mask_import Tvent {} \\\n".format(mask_Ventricles))
            #f.write("-mask_intersect Svent CSF Tvent \\\n")
            f.write("-dsets {} \\\n".format(imgFile))
            f.write("-regress_motion_file {} \\\n".format(path_motion))
            f.write("-regress_censor_extern {} \\\n".format(path_motion_outlier))
            #f.write("-regress_ROI Svent \\\n")
            f.write("-regress_ROI_PC CSF 5 \\\n")
            f.write("-regress_ROI_PC WM 5 \\\n")
            f.write("-regress_stim_times {} {} \\\n".format(timingFile_regressor,timingFile_missing))
            f.write("-regress_basis 'SPMG1' \\\n")
            f.write("-regress_stim_labels reg miss \\\n")
            f.write("-regress_opts_3dD -bout -jobs 6 \\\n")
            #f.write("-gltsym 'SYM: ROIPC . CSFe[0]' -glt_label 4 CSFe \\\n")
            #f.write("-gltsym 'MATRIX: 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0' -glt_label 4 CSFe \\\n")
            f.write("-regress_stim_types AM2 times \\\n")
            f.write("-regress_polort 4 \\\n")        
            #f.write("-regress_apply_mot_types demean \\\n")
            #f.write("-regress_motion_per_run \\\n")
            #f.write("-regress_3dD_stop \\\n")
            #f.write("-regress_reml_exec \\\n")
            f.write("-regress_make_ideal_sum sum_ideal.1D \\\n")
            f.write("-regress_run_clustsim no \\\n")
            f.write("\n")
            
    masterScript = os.path.join(Dir_firstlevel,subID,'run_firstlevel_{}.sh'.format(subID))
    
    if len(df_subj)!=0:
        with open(masterScript, "w") as f:
            f.write("#!/bin/tcsh\n")

            for sesID in df_subj.sesID:
                output_1st_subj = os.path.join(Dir_firstlevel,subID,sesID)
                scriptFile = os.path.join(output_1st_subj,'Batch_firstlevel_{}_{}.sh'.format(subID,sesID))
                f.write("cd {} \n".format(output_1st_subj))
                f.write("tcsh {} \n".format(scriptFile))
                f.write("tcsh proc.{} \n".format(subID))
                f.write("rm {}/{}.results/pb00.* \n".format(output_1st_subj,subID))
        print(subID,'generated correctly.')
    
    

sub-1006 generated correctly.
sub-1007 generated correctly.
sub-1008 generated correctly.
sub-1009 generated correctly.
check:sub-1011-visit01.
sub-1011 generated correctly.
sub-1012 generated correctly.
sub-1014 generated correctly.
sub-1016 generated correctly.
sub-1018 generated correctly.
sub-1019 generated correctly.
sub-1020 generated correctly.
sub-1021 generated correctly.
sub-1022 generated correctly.
sub-1023 generated correctly.
sub-1024 generated correctly.
sub-1025 generated correctly.
sub-1026 generated correctly.
sub-1028 generated correctly.
sub-1029 generated correctly.
sub-1030 generated correctly.
sub-1032 generated correctly.
sub-1036 generated correctly.
sub-1037 generated correctly.
sub-1039 generated correctly.
sub-1040 generated correctly.
sub-1041 generated correctly.
sub-1042 generated correctly.
sub-1044 generated correctly.
sub-1045 generated correctly.
sub-1047 generated correctly.
sub-1048 generated correctly.
sub-1049 generated correctly.
sub-1052 generat